# EXPERIMENT-5: POS Tagging, Chunking and Parsing

**Aim**

1. Categorize and tag words in Twitter data.
2. Implement POS tagging and chunking for sentences.
3. Demonstrate shallow parsing and dependency parsing.

**Dataset:** Kaggle Twitter Tweets Sentiment Dataset (`Tweets.csv`).

In [ ]:
# Install required packages
!pip -q install pandas matplotlib seaborn nltk spacy kagglehub
!python -m spacy download en_core_web_sm

In [ ]:
import os
import glob
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import kagglehub
import spacy

from nltk.tokenize import word_tokenize, TweetTokenizer
from nltk import pos_tag, RegexpParser

# Download NLTK resources required by current NLTK versions
resources = [
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng"
]

for resource in resources:
    nltk.download(resource, quiet=True)

print("Libraries and NLTK resources loaded successfully.")

## 1. Download and Load the Kaggle Dataset

In [ ]:
path = kagglehub.dataset_download("yasserh/twitter-tweets-sentiment-dataset")

csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)

print("Dataset location:", path)
print("\nCSV files found:")
for f in csv_files:
    print("-", f)

if not csv_files:
    raise FileNotFoundError("No CSV file found.")

tweet_file = next(
    (f for f in csv_files if os.path.basename(f).lower() == "tweets.csv"),
    csv_files[0]
)

df = pd.read_csv(tweet_file)

print("\nUsing file:", tweet_file)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Understand the Dataset

In [ ]:
print("Columns:")
print(list(df.columns))

print("\nMissing values:")
display(df.isnull().sum())

print("\nData types:")
display(df.dtypes)

In [ ]:
# Select the tweet/text column explicitly when available.
# The Kaggle dataset normally contains a column named text.
if "text" in df.columns:
    TEXT_COLUMN = "text"
else:
    candidates = [
        c for c in df.columns
        if any(k in c.lower() for k in ["text", "tweet", "content", "message"])
    ]
    if not candidates:
        raise ValueError("Could not identify the tweet/text column.")
    TEXT_COLUMN = candidates[0]

df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
df = df[df[TEXT_COLUMN].str.strip().ne("")].reset_index(drop=True)

print("Selected text column:", TEXT_COLUMN)
print("Usable tweets:", len(df))
display(df[[TEXT_COLUMN]].head(10))

## 3. Inspect Twitter Text

Twitter data may contain hashtags, mentions, URLs, punctuation, emojis and informal language.

In [ ]:
sample_tweets = df[TEXT_COLUMN].sample(min(10, len(df)), random_state=42).tolist()

print("Sample Twitter data:\n")
for i, tweet in enumerate(sample_tweets, 1):
    print(f"{i}. {tweet}")

## 4. Twitter Tokenization

`TweetTokenizer` is designed for social-media text and is therefore more appropriate than a simple whitespace split.

In [ ]:
tweet_tokenizer = TweetTokenizer(
    preserve_case=True,
    reduce_len=True,
    strip_handles=False
)

example_tweet = sample_tweets[0]
tokens = tweet_tokenizer.tokenize(example_tweet)

print("Tweet:")
print(example_tweet)

print("\nTokens:")
print(tokens)

## 5. POS Tagging of a Sentence

POS tagging assigns a grammatical category to each token.

Common Penn Treebank tags:

- NN — noun
- VB — verb
- JJ — adjective
- RB — adverb
- PRP — personal pronoun
- DT — determiner

In [ ]:
sentence = "The students are learning natural language processing today."

tokens = word_tokenize(sentence)
tags = pos_tag(tokens)

print("Sentence:", sentence)
print("\nPOS Tags:")
print(f"{'Word':20} {'POS Tag'}")
print("-" * 30)

for word, tag in tags:
    print(f"{word:20} {tag}")

### Expected output

```text
The                  DT
students             NNS
are                  VBP
learning             VBG
natural              JJ
language             NN
processing           NN
today                NN
.                    .
```

Exact output can vary slightly with the installed NLTK model version.

## 6. POS Tagging on Twitter Data

In [ ]:
N_TWEETS = min(500, len(df))
twitter_sample = df.sample(N_TWEETS, random_state=42).copy()

def twitter_pos_tag(text):
    tokens = tweet_tokenizer.tokenize(text)
    return pos_tag(tokens)

twitter_sample["tokens"] = twitter_sample[TEXT_COLUMN].apply(
    tweet_tokenizer.tokenize
)

twitter_sample["pos_tags"] = twitter_sample[TEXT_COLUMN].apply(
    twitter_pos_tag
)

print(f"POS tagging completed for {N_TWEETS} tweets.")
display(twitter_sample[[TEXT_COLUMN, "tokens", "pos_tags"]].head(10))

## 7. Categorize POS Tags in Twitter Data

In [ ]:
all_tags = []

for tagged_sentence in twitter_sample["pos_tags"]:
    all_tags.extend(tag for _, tag in tagged_sentence)

tag_counts = Counter(all_tags)

tag_df = (
    pd.DataFrame(
        tag_counts.items(),
        columns=["POS_Tag", "Frequency"]
    )
    .sort_values("Frequency", ascending=False)
    .reset_index(drop=True)
)

print("Top POS tags:")
display(tag_df.head(20))

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=tag_df.head(15),
    x="Frequency",
    y="POS_Tag"
)
plt.title("Top POS Tags in Twitter Sample")
plt.xlabel("Frequency")
plt.ylabel("POS Tag")
plt.tight_layout()
plt.show()

## 8. Broad POS Categories

In [ ]:
def broad_category(tag):
    if tag.startswith("NN"):
        return "Noun"
    if tag.startswith("VB"):
        return "Verb"
    if tag.startswith("JJ"):
        return "Adjective"
    if tag.startswith("RB") or tag == "WRB":
        return "Adverb"
    if tag in {"PRP", "WP"}:
        return "Pronoun"
    if tag in {"PRP$", "WP$"}:
        return "Possessive Pronoun"
    if tag in {"DT", "PDT", "WDT"}:
        return "Determiner"
    if tag in {"IN", "TO"}:
        return "Preposition/Subordinator"
    if tag == "CC":
        return "Conjunction"
    if tag == "CD":
        return "Number"
    if tag == "MD":
        return "Modal"
    if tag == "UH":
        return "Interjection"
    if tag == "RP":
        return "Particle"
    if tag == "POS":
        return "Possessive"
    if tag == "SYM":
        return "Symbol"
    if tag in {".", ",", ":", "``", "''", "-LRB-", "-RRB-"}:
        return "Punctuation"
    return "Other"

category_counts = Counter(
    broad_category(tag)
    for tag in all_tags
)

category_df = (
    pd.DataFrame(
        category_counts.items(),
        columns=["Category", "Frequency"]
    )
    .sort_values("Frequency", ascending=False)
    .reset_index(drop=True)
)

display(category_df)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=category_df,
    x="Frequency",
    y="Category"
)
plt.title("Broad POS Categories in Twitter Data")
plt.xlabel("Frequency")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

## 9. POS Tagging and Dependency Parsing with spaCy

In [ ]:
nlp = spacy.load("en_core_web_sm")

test_sentence = (
    "The intelligent student solved the difficult NLP problem quickly."
)

doc = nlp(test_sentence)

print(f"{'Token':15} {'POS':10} {'Tag':10} {'Dependency':15} {'Head':15}")
print("-" * 70)

for token in doc:
    print(
        f"{token.text:15} "
        f"{token.pos_:10} "
        f"{token.tag_:10} "
        f"{token.dep_:15} "
        f"{token.head.text:15}"
    )

## 10. Chunking / Shallow Parsing

Chunking groups words into meaningful phrases.

The grammar below identifies:

- **NP** — Noun Phrase
- **VP** — Verb Phrase
- **PP** — Prepositional Phrase

In [ ]:
# Improved chunk grammar
chunk_grammar = r'''
    NP: {<DT|PRP\$>?<JJ.*>*<NN.*>+}
    NP: {<NNP>+}
    VP: {<MD>?<VB.*>(<RB.*>)*}
    PP: {<IN|TO><NP>}
'''

chunk_parser = RegexpParser(chunk_grammar)

chunk_sentence = (
    "The intelligent student solved the difficult NLP problem "
    "in the laboratory."
)

chunk_tokens = word_tokenize(chunk_sentence)
chunk_tagged = pos_tag(chunk_tokens)
chunk_tree = chunk_parser.parse(chunk_tagged)

print("Sentence:")
print(chunk_sentence)

print("\nPOS tagged sentence:")
print(chunk_tagged)

print("\nChunked structure:")
print(chunk_tree)

### Expected chunk interpretation

The important phrase-level structures should include approximately:

- **NP:** The intelligent student
- **NP:** the difficult NLP problem
- **PP:** in the laboratory

The exact tree representation depends on the NLTK POS tagger version.

In [ ]:
print("Extracted Noun Phrases:")
print("-" * 35)

for subtree in chunk_tree.subtrees(
    filter=lambda t: t.label() == "NP"
):
    phrase = " ".join(word for word, tag in subtree.leaves())
    print("NP:", phrase)

## 11. Chunking a Twitter-Style Sentence

In [ ]:
twitter_sentence = (
    "The new AI model is really amazing for image classification!"
)

twitter_tagged = pos_tag(
    tweet_tokenizer.tokenize(twitter_sentence)
)

twitter_tree = chunk_parser.parse(twitter_tagged)

print("Tweet:")
print(twitter_sentence)

print("\nPOS Tags:")
print(twitter_tagged)

print("\nChunks:")
print(twitter_tree)

print("\nExtracted Noun Phrases:")
for subtree in twitter_tree.subtrees(
    filter=lambda t: t.label() == "NP"
):
    print("NP:", " ".join(word for word, tag in subtree.leaves()))

## 12. Dependency Parsing

In [ ]:
dependency_sentence = (
    "The researcher developed a new NLP model for Twitter data."
)

dependency_doc = nlp(dependency_sentence)

print(f"{'Token':15} {'POS':10} {'Dependency':15} {'Head':15}")
print("-" * 60)

for token in dependency_doc:
    print(
        f"{token.text:15} "
        f"{token.pos_:10} "
        f"{token.dep_:15} "
        f"{token.head.text:15}"
    )

## 13. Visualize Dependency Parse

In [ ]:
from spacy import displacy

displacy.render(
    dependency_doc,
    style="dep",
    jupyter=True,
    options={"distance": 100}
)

## 14. POS Tagging of Multiple Twitter Samples

In [ ]:
print("POS tagging of five Twitter samples:\n")

for i, tweet in enumerate(
    twitter_sample[TEXT_COLUMN].head(5), 1
):
    print(f"Tweet {i}:")
    print(tweet)
    print("Tags:", twitter_pos_tag(tweet))
    print("-" * 80)

## 15. Experiment Observations

1. POS tagging assigns grammatical categories to individual tokens.
2. Twitter text is noisy because it may contain hashtags, mentions, URLs, emojis and informal spellings.
3. `TweetTokenizer` is suitable for social-media text.
4. NLTK provides POS tagging and grammar-based chunking.
5. Chunking identifies shallow phrase structures such as noun phrases and prepositional phrases.
6. spaCy provides POS tagging together with dependency relations and visualization.
7. POS taggers trained on standard English may be less accurate on informal Twitter language.

## 16. Result

POS tagging was successfully applied to Twitter data. The experiment also demonstrated tokenization, POS-tag frequency analysis, broad grammatical categorization, chunking/shallow parsing and dependency parsing.

The notebook uses the Kaggle Twitter dataset for real-world Twitter text and controlled sentences for demonstrating phrase and dependency structures.

## Viva Questions

1. What is POS tagging?
2. Why is POS tagging useful in NLP?
3. What is the difference between POS tagging and chunking?
4. What is a noun phrase?
5. What is a verb phrase?
6. Why is Twitter POS tagging difficult?
7. What is shallow parsing?
8. What is dependency parsing?
9. What is the difference between NLTK and spaCy?
10. What are NN, VB, JJ, RB and DT?
11. What is the purpose of a chunk grammar?
12. What is the difference between a POS tag and a dependency relation?